# Audited European Options: Prices, Greeks and Payoffs

This notebook illustrates a **locally tested Black–Scholes–Merton model** and expiry strategies. It replaces unvalidated calculations in the earlier interactive notebook.

**Inputs are illustrative**, not Kotak Mahindra observed option premiums or any exchange backtest. Run from the repository root after installing `options_pricing/requirements.txt`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from options_pricing.engine import BSM, expiry_profit, sample_strategies

## European call and put pricing: put–call parity and Greeks

In [ ]:
m = BSM(spot=100, strike=100, years=1.0, rate=0.05, volatility=0.20)
call, put = m.price("call"), m.price("put")
assert np.isclose(call - put, m.spot - m.strike * np.exp(-m.rate * m.years))
print("Call:", round(call, 6), "| Put:", round(put, 6))
print("Call Greeks:", m.greeks("call"))
print("Put Greeks:", m.greeks("put"))

## Multi-leg expiry profit: premium signs validated

In [ ]:
prices = np.linspace(60, 140, 201)
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, name in zip(axes.flat, ("bull_call_spread", "long_straddle", "long_call_butterfly", "covered_call")):
    legs, shares = sample_strategies()[name]
    pnl = expiry_profit(legs, prices, initial_spot=100, stock_quantity=shares)
    ax.plot(prices, pnl)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_title(name.replace("_", " ").title())
    ax.set_ylabel("Expiry P&L per underlying unit")
    ax.grid(alpha=0.3)
for ax in axes[-1]: ax.set_xlabel("Underlying price at expiry")
plt.tight_layout()
plt.show()

## Interpret carefully

A purchased call below its strike loses its **paid premium**. The original downloaded notebook had the opposite sign, and also displayed misleading "maximum profit/loss" labels derived from only a finite spot-price grid. Both problems are avoided here.

These are **European exercise, per-unit analytical illustrations**. Actual market work requires exchange contract specifications, dividend/interest conventions, lot sizes, observed option premiums, liquidity costs, exercise rules and real data.